# TB Portals â€” 03 Â· Train baseline (Day-1 GATE)

Reproduce Kantipudi A2: single DenseNet121 ALP regressor + cavity classifier, NAdam 1e-3, country-segregated. **Enable Internet** in Kaggle (DenseNet121 ImageNet weights download once).

Strategy on a T4 (9â€“12h sessions): run **1 seed Ã— 3 countries first** to hit the gate, then add seeds. Training appends to `results.csv` and is `--resume`-able, so you can split across sessions.

In [ ]:
# Pull latest codebase from GitHub
import os, subprocess, sys
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
if os.path.isdir(REPO_DIR):
subprocess.run(["git","-C",REPO_DIR,"pull","--ff-only"],check=True)
else:
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("repo ready at", REPO_DIR)


In [ ]:
import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
# Ensure repo is cloned and on sys.path
if not os.path.isdir(REPO_DIR):
    import subprocess
subprocess.run(["git","clone","--depth","1","--branch","cleaned-repo","https://github.com/mabdullahi7780/dl-project-codebase.git",REPO_DIR],check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import sys, os
REPO_DIR = '/kaggle/working/dl-project-codebase'
WORK     = '/kaggle/working'
MANIFEST = f'{WORK}/data/processed/tbportals_manifest.csv'
OUT_DIR  = f'{WORK}/checkpoints/tbportals/baseline'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, REPO_DIR)
from src.training.train_tbportals_baseline import main as train_main
print('manifest:', MANIFEST)
print('out_dir: ', OUT_DIR)


## Step 1 — Smoke test (~2 min)
Verifies the pipeline end-to-end before committing to a full run.

In [ ]:
# 2 epochs, Romania held out — should complete in ~2 min on Kaggle GPU
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania',
    '--seeds',       '0',
    '--epochs',      '2',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--out-dir',     f'{WORK}/checkpoints/tbportals/smoke',
])


## Step 2 — Gate run (~90 min on P100/T4)
3 held-out countries x seed 0 x 50 epochs. Must pass before MoE work.

In [ ]:
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '0',
    '--epochs',      '50',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--amp',
    '--out-dir',     OUT_DIR,
])


## Step 3 — Multi-seed run (seeds 1 & 2)
Run after gate passes to get mean +/- std for the paper.

In [ ]:
train_main([
    '--manifest',    MANIFEST,
    '--held-outs',   'Romania', 'Moldova', 'Kazakhstan',
    '--seeds',       '1', '2',
    '--epochs',      '50',
    '--batch-size',  '32',
    '--num-workers', '2',
    '--amp',
    '--out-dir',     OUT_DIR,
])
